In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from us import states
import requests

import os
from dotenv import load_dotenv

from sklearn.preprocessing import MinMaxScaler

# Downloading ACS Data

In [ ]:
load_dotenv()
key = os.environ.get('CENSUS_API_KEY')

get = 'NAME,S0101_C01_001E,S2506_C01_024E,S0102_C01_037E,S2101_C01_034E,S2701_C01_001E,S2503_C01_028E'
url = f"https://api.census.gov/data/{year}/acs/acs5/subject?get={get}&for=congressional%20district:*&in=state:*&key={key}"

In [ ]:
for year in [2020, 2021, 2022, 2023, 2024]:
    url = f"https://api.census.gov/data/{year}/acs/acs5/subject?get={get}&for=congressional%20district:*&in=state:*&key={key}"
    response = requests.get(url)
    df = pd.DataFrame(response.json(), columns=response.json()[0])[1:]
    df.rename({
        'NAME': 'name',
        'S0101_C01_001E': 'population',
        'S2506_C01_024E': 'income',
        'S0102_C01_037E': 'educational attainment',
        'S2101_C01_034E': 'unemployment',
        'S2701_C01_001E': 'health insurance coverage',
        'S2503_C01_028E': 'housing cost burden'
    }, axis = 1, inplace=True)

    df.to_csv(f"Raw ACS Data/acs_{year}.csv", index=False)

# Preparing Datasets

In [ ]:
# importing data and converting the state fips codes to state names
df = pd.read_csv("Raw ACS Data/acs_2024.csv")
df['state_name'] = df['state'].astype(str).str.zfill(2).apply(lambda x: states.lookup(x))

In [84]:
# scaling the data
scaler = MinMaxScaler()
df[['population', 'income', 'educational attainment', 'unemployment', 'health insurance coverage', 'housing cost burden']] = scaler.fit_transform(df[['population', 'income', 'educational attainment', 'unemployment', 'health insurance coverage', 'housing cost burden']])

# flipping unemployment and housing burden, so high values indicate better conditions
df['unemployment'] = 1 - df['unemployment']
df['housing cost burden'] = 1 - df['housing cost burden']

#calculating a compisite score
df['composite_score'] = (df['population'] + df['income'] + df['educational attainment'] + df['unemployment'] + df['health insurance coverage'] + df['housing cost burden']) / 6

In [20]:
for file in os.listdir("Raw ACS Data"):
    print(f"Processing {file}")
    if file.endswith(".csv"):
        # reading in data
        df = pd.read_csv(f"Raw ACS Data/{file}")

        # converting state codes to full state names
        df['state_name'] = df['state'].astype(str).str.zfill(2).apply(lambda x: states.lookup(x))

        # scaling the data
        scaler = MinMaxScaler()
        df[['income', 'educational attainment', 'unemployment', 'health insurance coverage', 'housing cost burden']] = scaler.fit_transform(df[['income', 'educational attainment', 'unemployment', 'health insurance coverage', 'housing cost burden']])

        # flipping unemployment and housing burden, so high values indicate better conditions
        df['unemployment'] = 1 - df['unemployment']
        df['housing cost burden'] = 1 - df['housing cost burden']

        #calculating a compisite score
        df['composite_score'] = (df['income'] + df['educational attainment'] + df['unemployment'] + df['health insurance coverage'] + df['housing cost burden']) / 6


        df.to_csv(f"Processed ACS Data/{file}", index=False)

Processing acs_2020.csv
Processing acs_2021.csv
Processing acs_2022.csv
Processing acs_2023.csv
Processing acs_2024.csv


# Downloading Sharefiles

In [3]:
# downloading CD118 Shapefiles
url = "https://www2.census.gov/geo/tiger/TIGER2023/CD/"

for i in range(1, 80):
    response = requests.get(url + f"tl_2023_{str(i).zfill(2)}_cd118.zip")
    print(response.status_code)

    if response.status_code == 200:
        with open(f"Shapefiles/cd118/tl_2023_{str(i).zfill(2)}_cd118.zip", "wb") as f:
            f.write(response.content)
            print(f"Downloaded tl_2023_{str(i).zfill(2)}_cd118.zip")

200
Downloaded tl_2023_01_cd118.zip
200
Downloaded tl_2023_02_cd118.zip
404
200
Downloaded tl_2023_04_cd118.zip
200
Downloaded tl_2023_05_cd118.zip
200
Downloaded tl_2023_06_cd118.zip
404
200
Downloaded tl_2023_08_cd118.zip
200
Downloaded tl_2023_09_cd118.zip
200
Downloaded tl_2023_10_cd118.zip
200
Downloaded tl_2023_11_cd118.zip
200
Downloaded tl_2023_12_cd118.zip
200
Downloaded tl_2023_13_cd118.zip
404
200
Downloaded tl_2023_15_cd118.zip
200
Downloaded tl_2023_16_cd118.zip
200
Downloaded tl_2023_17_cd118.zip
200
Downloaded tl_2023_18_cd118.zip
200
Downloaded tl_2023_19_cd118.zip
200
Downloaded tl_2023_20_cd118.zip
200
Downloaded tl_2023_21_cd118.zip
200
Downloaded tl_2023_22_cd118.zip
200
Downloaded tl_2023_23_cd118.zip
200
Downloaded tl_2023_24_cd118.zip
200
Downloaded tl_2023_25_cd118.zip
200
Downloaded tl_2023_26_cd118.zip
200
Downloaded tl_2023_27_cd118.zip
200
Downloaded tl_2023_28_cd118.zip
200
Downloaded tl_2023_29_cd118.zip
200
Downloaded tl_2023_30_cd118.zip
200
Downloaded t

In [ ]:
# downloading CD119 Shapefiles
url = "https://www2.census.gov/geo/tiger/TIGER2024/CD/"

for i in range(1, 78):
    response = requests.get(url + f"tl_2024_{str(i).zfill(2)}_cd119.zip")
    print(response.status_code)

    if response.status_code == 200:
        with open(f"Shapefiles/cd119/tl_2024_{str(i).zfill(2)}_cd119.zip", "wb") as f:
            f.write(response.content)
            print(f"Downloaded tl_2024_{str(i).zfill(2)}_cd119.zip")

200
Downloaded tl_2024_01_cd119.zip
200
Downloaded tl_2024_02_cd119.zip
404
200
Downloaded tl_2024_04_cd119.zip
200
Downloaded tl_2024_05_cd119.zip
200
Downloaded tl_2024_06_cd119.zip
404
200
Downloaded tl_2024_08_cd119.zip
200
Downloaded tl_2024_09_cd119.zip
200
Downloaded tl_2024_10_cd119.zip
200
Downloaded tl_2024_11_cd119.zip
200
Downloaded tl_2024_12_cd119.zip
200
Downloaded tl_2024_13_cd119.zip
404
200
Downloaded tl_2024_15_cd119.zip
200
Downloaded tl_2024_16_cd119.zip
200
Downloaded tl_2024_17_cd119.zip
200
Downloaded tl_2024_18_cd119.zip
200
Downloaded tl_2024_19_cd119.zip
200
Downloaded tl_2024_20_cd119.zip
200
Downloaded tl_2024_21_cd119.zip
200
Downloaded tl_2024_22_cd119.zip
200
Downloaded tl_2024_23_cd119.zip
200
Downloaded tl_2024_24_cd119.zip
200
Downloaded tl_2024_25_cd119.zip
200
Downloaded tl_2024_26_cd119.zip
200
Downloaded tl_2024_27_cd119.zip
200
Downloaded tl_2024_28_cd119.zip
200
Downloaded tl_2024_29_cd119.zip
200
Downloaded tl_2024_30_cd119.zip
200
Downloaded t

# Appending ACS data to shapefile

## For years 2020 through 2023 (CD118)

In [4]:
for year in [2020, 2021, 2022, 2023]:
    df = pd.read_csv(f"Processed ACS Data/acs_{year}.csv")
    df['GEOID'] = df['state'].astype(str).str.zfill(2) + df['congressional district'].astype(str)

    complete_gdf = gpd.GeoDataFrame()

    for shapefile in os.listdir("Shapefiles/cd118"):
        gdf = gpd.read_file(f"Shapefiles/cd118/{shapefile}")
        gdf_acs = gdf.merge(df, on='GEOID', how='left')

        complete_gdf = pd.concat([complete_gdf, gdf_acs], ignore_index=True)

    complete_gdf['geometry'] = complete_gdf['geometry'].simplify(tolerance=0.01, preserve_topology=True)
    complete_gdf.to_parquet(f"Processed ACS Data/withGeo/acs_{year}_with_geometries.parquet")

## For year 2024 (CD119)

In [7]:
df = pd.read_csv("Processed ACS Data/acs_2024.csv")
df['GEOID'] = df['state'].astype(str).str.zfill(2) + df['congressional district'].astype(str)

complete_gdf = gpd.GeoDataFrame()

for shapefile in os.listdir("Shapefiles/cd119"):
    gdf = gpd.read_file(f"Shapefiles/cd119/{shapefile}")
    gdf_acs = gdf.merge(df, on='GEOID', how='left')

    complete_gdf = pd.concat([complete_gdf, gdf_acs], ignore_index=True)

complete_gdf['geometry'] = complete_gdf['geometry'].simplify(tolerance=0.01, preserve_topology=True)
complete_gdf.to_parquet("Processed ACS Data/withGeo/acs_2024_with_geometries.parquet")


In [22]:
complete_gdf.to_file("Processed ACS Data/Shapefiles/2024/acs_2024_with_geometries.shp", index=False)
complete_gdf.to_parquet("Processed ACS Data/Shapefiles/2024/acs_2024_with_geometries.parquet")

C:\Users\wlvau\AppData\Local\Temp\ipykernel_16468\3704586325.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  complete_gdf.to_file("Processed ACS Data/Shapefiles/2024/acs_2024_with_geometries.shp", index=False)
c:\Users\wlvau\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'educational attainment' to 'educationa'
  ogr_write(
c:\Users\wlvau\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'unemployment' to 'unemployme'
  ogr_write(
c:\Users\wlvau\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'health insurance coverage' to 'health ins'
  ogr_write(
c:\Users\wlvau\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'housin

#Converting to parquet for speed